# 05 - OpenAI Embeddings, Chroma Vector Store, and Retriever

## Purpose

This notebook creates the vector search layer for Version 2 of the PDF Q&A chatbot.

The cleaned and token-split PDF documents are converted into OpenAI embeddings and stored in a Chroma vector database.

This notebook also creates a retriever that can search the vector database and return the most relevant document chunks for a user question.

## Main Changes in Version 2

This version adds:

- OpenAI embeddings instead of Hugging Face embeddings
- Chroma instead of FAISS
- Token-split documents as the indexing input
- MMR retrieval for more diverse search results
- Persistent vector database storage

## Main Outputs

- `embedding`
- `vectorstore`
- `retriever`
- Chroma database path: `/Volumes/workspace/365pdf/365pdfupload/chroma_intro_tableau_v2`

## Notebook Flow

```text
Token-split PDF documents
        ↓
OpenAI embedding model
        ↓
Chroma vector database
        ↓
MMR retriever
        ↓
Semantic search test
        ↓
Ready for Q&A chain

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os

os.environ["OPENAI_API_KEY"] = "OpenAI_API_KEY"

In [0]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, TokenTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
import time

pdf_path = "/Volumes/workspace/365pdf/365pdfupload/Introduction_to_Tableau.pdf"

loader_pdf = PyPDFLoader(pdf_path)
docs_list = loader_pdf.load()

markdown_transcript = ""

for doc in docs_list:
    page_number = doc.metadata.get("page", "unknown")
    page_text = doc.page_content

    markdown_transcript += f"\n\n# Introduction to Tableau\n"
    markdown_transcript += f"## Page {page_number}\n\n"
    markdown_transcript += page_text

headers_to_split_on = [
    ("#", "Section Title"),
    ("##", "Page Title")
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs_list_md_split = md_splitter.split_text(markdown_transcript)

chat_formatter = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

formatting_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a document-cleaning assistant.

Your task is to clean text extracted from a PDF.

Rules:
- Fix broken line breaks and awkward spacing.
- Improve punctuation and capitalization where needed.
- Keep the original meaning unchanged.
- Do not add new information.
- Do not summarize.
- Do not remove important details.
- Return only the cleaned text.
"""
    ),
    (
        "human",
        """
Clean and format the following extracted PDF text:

{text}
"""
    )
])

formatting_chain = formatting_prompt | chat_formatter | StrOutputParser()

formatted_docs_list = []

for i, doc in enumerate(docs_list_md_split):
    print(f"Formatting document {i + 1} of {len(docs_list_md_split)}")

    text_to_format = doc.page_content[:3000]

    formatted_text = formatting_chain.invoke({
        "text": text_to_format
    })

    formatted_doc = Document(
        page_content=formatted_text,
        metadata=doc.metadata
    )

    formatted_docs_list.append(formatted_doc)

    time.sleep(0.5)

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

docs_list_tokens_split = token_splitter.split_documents(formatted_docs_list)

print("Token-split documents ready:", len(docs_list_tokens_split))

In [0]:
for i, doc in enumerate(docs_list_tokens_split[:3]):
    print(f"\nDocument {i}")
    print("Metadata:", doc.metadata)
    print("Preview:")
    print(doc.page_content[:700])
    print("-" * 80)

In [0]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("OpenAI embedding model ready")

In [0]:
test_vector = embedding.embed_query("What is Tableau used for?")

print("Embedding created successfully")
print("Vector length:", len(test_vector))
print("First 10 values:", test_vector[:10])

In [0]:
from langchain_community.vectorstores import Chroma
import shutil
import os

persist_directory = "/tmp/chroma_intro_tableau_v2"

if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
    print("Removed existing Chroma directory")

vectorstore = Chroma.from_documents(
    documents=docs_list_tokens_split,
    embedding=embedding,
    persist_directory=persist_directory
)

print("Chroma vector database created")
print("Persist directory:", persist_directory)

In [0]:
try:
    vectorstore.persist()
    print("Chroma vector database persisted")
except Exception as e:
    print("Persist method not required or not available:", e)

In [0]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 2,
        "fetch_k": 6,
        "lambda_mult": 0.7
    }
)

print("MMR retriever created successfully")

In [0]:
question = "What is Tableau used for?"

retrieved_docs = retriever.invoke(question)

print("Question:", question)
print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nRetrieved Document {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:1000])
    print("-" * 80)

In [0]:
question = "What are dimensions and measures in Tableau?"

retrieved_docs = retriever.invoke(question)

print("Question:", question)
print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nRetrieved Document {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:1000])
    print("-" * 80)

In [0]:
loaded_vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

loaded_retriever = loaded_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 2,
        "fetch_k": 6,
        "lambda_mult": 0.7
    }
)

print("Loaded Chroma vector database successfully")

In [0]:
question = "How does Tableau help with data visualization?"

loaded_docs = loaded_retriever.invoke(question)

for i, doc in enumerate(loaded_docs, start=1):
    print(f"\nLoaded Result {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:800])
    print("-" * 80)